# DS1/GSE103224 — build the raw-count AnnData object

This notebook reconstructs the eight-sample GSE103224 count object from the
GEO supplementary files matching:

`GSM*_PJ*.filtered.matrix.txt.gz`

Each source file is a headerless, tab-separated gene-by-cell matrix in which:

1. column 1 is the versioned Ensembl gene identifier;
2. column 2 is the gene symbol;
3. remaining columns are raw integer counts for individual cells.

The parser reads the wide matrices in row chunks to avoid loading an entire
sample into a dense pandas object at once. Synthetic cell identifiers are
generated because the GEO files do not contain cell-barcode headers.

## Expected result

- 8 samples
- 23,793 cells
- 60,725 Ensembl features

## Default paths

Input: `data/raw/GSE103224/`

Output:
`data/interim/DS1_GSE103224/GSE103224_all_samples_inner.h5ad`

The paths may be overridden with `GLIOMA_PROJECT_ROOT`,
`GSE103224_RAW_DIR`, and `GSE103224_INTERIM_H5AD`.

In [ ]:
import gc
import gzip
import json
import os
import re
import sys
from datetime import datetime
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

print("Python:", sys.version)
print("anndata:", ad.__version__)
print("scanpy:", sc.__version__)
print("pandas:", pd.__version__)

In [ ]:
def find_repository_root(start: Path) -> Path:
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root(Path.cwd()))
).expanduser().resolve()

RAW_DIR = Path(
    os.environ.get(
        "GSE103224_RAW_DIR",
        PROJECT_ROOT / "data" / "raw" / "GSE103224",
    )
).expanduser().resolve()

OUTPUT_H5AD = Path(
    os.environ.get(
        "GSE103224_INTERIM_H5AD",
        PROJECT_ROOT
        / "data"
        / "interim"
        / "DS1_GSE103224"
        / "GSE103224_all_samples_inner.h5ad",
    )
).expanduser().resolve()

AUDIT_DIR = (
    PROJECT_ROOT / "results" / "qc" / "DS1_GSE103224" / "00a_build"
)

CHUNK_ROWS = int(os.environ.get("GSE103224_CHUNK_ROWS", "250"))

OUTPUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_FILES = {
    "GSM2758471": ("PJ016", 3085),
    "GSM2758472": ("PJ017", 1261),
    "GSM2758473": ("PJ018", 2197),
    "GSM2758474": ("PJ025", 5924),
    "GSM2758475": ("PJ030", 3097),
    "GSM2758476": ("PJ032", 1377),
    "GSM2758477": ("PJ035", 3768),
    "GSM2940098": ("PJ048", 3084),
}
EXPECTED_N_GENES = 60_725
EXPECTED_TOTAL_CELLS = 23_793

print("Project root:", PROJECT_ROOT)
print("RAW directory:", RAW_DIR)
print("Output:", OUTPUT_H5AD)
print("Chunk rows:", CHUNK_ROWS)

if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"RAW directory not found: {RAW_DIR}\n"
        "Place the eight GEO matrix files there or set GSE103224_RAW_DIR."
    )

In [ ]:
FILE_PATTERN = re.compile(
    r"^(GSM\d+)_(PJ\d+)\.filtered\.matrix\.txt\.gz$"
)


def discover_input_files(raw_dir: Path) -> list[dict]:
    records = []
    for path in sorted(raw_dir.glob("GSM*_*.filtered.matrix.txt.gz")):
        match = FILE_PATTERN.match(path.name)
        if match is None:
            continue
        geo_sample_id, sample_id = match.groups()
        records.append(
            {
                "path": path,
                "geo_sample_id": geo_sample_id,
                "sample_id": sample_id,
            }
        )

    observed = {
        record["geo_sample_id"]: record["sample_id"] for record in records
    }
    expected = {
        geo_sample_id: values[0]
        for geo_sample_id, values in EXPECTED_FILES.items()
    }

    if observed != expected:
        missing = sorted(set(expected) - set(observed))
        extra = sorted(set(observed) - set(expected))
        mismatched = {
            key: (expected.get(key), observed.get(key))
            for key in set(expected) & set(observed)
            if expected[key] != observed[key]
        }
        raise ValueError(
            "The discovered GSE103224 files do not match the expected "
            f"eight-sample set. Missing={missing}, extra={extra}, "
            f"mismatched={mismatched}"
        )

    return records


input_records = discover_input_files(RAW_DIR)
pd.DataFrame(
    [
        {
            "geo_sample_id": r["geo_sample_id"],
            "sample_id": r["sample_id"],
            "filename": r["path"].name,
            "expected_cells": EXPECTED_FILES[r["geo_sample_id"]][1],
        }
        for r in input_records
    ]
)

In [ ]:
def _numeric_count_array(frame: pd.DataFrame, source: Path) -> np.ndarray:
    values = frame.to_numpy(copy=False)

    if not np.issubdtype(values.dtype, np.number):
        frame = frame.apply(pd.to_numeric, errors="raise")
        values = frame.to_numpy(copy=False)

    if np.issubdtype(values.dtype, np.floating):
        if not np.isfinite(values).all():
            raise ValueError(f"Non-finite count value detected in {source}")
        if not np.equal(values, np.floor(values)).all():
            raise ValueError(f"Non-integer count value detected in {source}")

    if np.any(values < 0):
        raise ValueError(f"Negative count value detected in {source}")

    maximum = values.max(initial=0)
    if maximum > np.iinfo(np.int32).max:
        raise OverflowError(
            f"A count in {source} exceeds the int32 range: {maximum}"
        )

    return values.astype(np.int32, copy=False)


def read_geo_matrix_chunked(
    path: Path,
    geo_sample_id: str,
    sample_id: str,
    chunk_rows: int = 250,
) -> ad.AnnData:
    gene_ids_versioned = []
    gene_symbols = []
    matrix_blocks = []
    n_cells = None

    reader = pd.read_csv(
        path,
        sep="\t",
        header=None,
        compression="gzip",
        chunksize=chunk_rows,
        low_memory=False,
    )

    for chunk_number, chunk in enumerate(reader, start=1):
        if chunk.shape[1] < 3:
            raise ValueError(
                f"{path.name}, chunk {chunk_number}: expected at least "
                "three columns."
            )

        current_n_cells = chunk.shape[1] - 2
        if n_cells is None:
            n_cells = current_n_cells
        elif current_n_cells != n_cells:
            raise ValueError(
                f"Inconsistent cell-column count in {path.name}: "
                f"{current_n_cells} versus {n_cells}"
            )

        gene_ids_versioned.extend(chunk.iloc[:, 0].astype(str).tolist())
        gene_symbols.extend(chunk.iloc[:, 1].astype(str).tolist())

        counts = _numeric_count_array(chunk.iloc[:, 2:], path)
        matrix_blocks.append(sp.csr_matrix(counts.T))

        del chunk, counts
        gc.collect()

    if n_cells is None or not matrix_blocks:
        raise ValueError(f"No matrix rows were read from {path}")

    X = sp.hstack(matrix_blocks, format="csr").astype(np.int32, copy=False)
    del matrix_blocks
    gc.collect()

    ensembl_no_version = pd.Series(gene_ids_versioned).str.replace(
        r"\.\d+$", "", regex=True
    )

    if ensembl_no_version.duplicated().any():
        duplicates = (
            ensembl_no_version[ensembl_no_version.duplicated(keep=False)]
            .drop_duplicates()
            .head(10)
            .tolist()
        )
        raise ValueError(
            "Removing Ensembl version suffixes produced duplicate "
            f"feature identifiers in {path.name}: {duplicates}"
        )

    obs_names = pd.Index(
        [f"{sample_id}_cell{i:06d}" for i in range(n_cells)],
        name="cell_id",
    )
    obs = pd.DataFrame(index=obs_names)
    obs["sample_id"] = sample_id
    obs["batch"] = sample_id
    obs["geo_sample_id"] = geo_sample_id
    obs["geo_id"] = "GSE103224"
    obs["core_dataset"] = "DS1_GSE103224"

    var = pd.DataFrame(
        {
            "ensembl_id_version": gene_ids_versioned,
            "gene_symbol": gene_symbols,
        },
        index=pd.Index(ensembl_no_version, name="ensembl_id"),
    )

    sample = ad.AnnData(X=X, obs=obs, var=var)
    sample.uns["source_file"] = path.name
    sample.uns["source_geo_accession"] = "GSE103224"

    return sample

In [ ]:
sample_objects = []
sample_summary_rows = []
reference_var = None

for record in input_records:
    geo_sample_id = record["geo_sample_id"]
    sample_id = record["sample_id"]
    expected_cells = EXPECTED_FILES[geo_sample_id][1]

    print(f"Reading {record['path'].name} ...")
    sample = read_geo_matrix_chunked(
        path=record["path"],
        geo_sample_id=geo_sample_id,
        sample_id=sample_id,
        chunk_rows=CHUNK_ROWS,
    )

    if sample.n_obs != expected_cells:
        raise AssertionError(
            f"{sample_id}: expected {expected_cells} cells, "
            f"observed {sample.n_obs}"
        )
    if sample.n_vars != EXPECTED_N_GENES:
        raise AssertionError(
            f"{sample_id}: expected {EXPECTED_N_GENES} genes, "
            f"observed {sample.n_vars}"
        )

    current_var = sample.var[
        ["ensembl_id_version", "gene_symbol"]
    ].copy()

    if reference_var is None:
        reference_var = current_var
    elif not current_var.equals(reference_var):
        raise ValueError(
            f"Gene identifiers/order in {record['path'].name} do not "
            "exactly match the first sample."
        )

    sample_summary_rows.append(
        {
            "geo_sample_id": geo_sample_id,
            "sample_id": sample_id,
            "cells": sample.n_obs,
            "genes": sample.n_vars,
            "total_counts": int(sample.X.sum()),
            "maximum_count": int(sample.X.max()),
            "source_file": record["path"].name,
        }
    )
    sample_objects.append(sample)
    print(sample)

sample_summary = pd.DataFrame(sample_summary_rows)
sample_summary.to_csv(
    AUDIT_DIR / "GSE103224_raw_build_sample_summary.tsv",
    sep="\t",
    index=False,
)
sample_summary

In [ ]:
adata_all = ad.concat(
    sample_objects,
    axis="obs",
    join="inner",
    merge="same",
    index_unique=None,
)
adata_all.var = reference_var.loc[adata_all.var_names].copy()

if not adata_all.obs_names.is_unique:
    raise AssertionError("Merged cell identifiers are not unique.")
if adata_all.n_obs != EXPECTED_TOTAL_CELLS:
    raise AssertionError(
        f"Expected {EXPECTED_TOTAL_CELLS} total cells, "
        f"observed {adata_all.n_obs}"
    )
if adata_all.n_vars != EXPECTED_N_GENES:
    raise AssertionError(
        f"Expected {EXPECTED_N_GENES} genes, observed {adata_all.n_vars}"
    )
if not sp.issparse(adata_all.X):
    raise TypeError("The merged count matrix is unexpectedly dense.")

nonzero_values = adata_all.X.data
if np.any(nonzero_values < 0):
    raise AssertionError("Negative counts detected after concatenation.")
if not np.equal(nonzero_values, np.floor(nonzero_values)).all():
    raise AssertionError("Non-integer counts detected after concatenation.")

adata_all.uns["dataset_id"] = "DS1_GSE103224"
adata_all.uns["geo_accession"] = "GSE103224"
adata_all.uns["build_timestamp"] = datetime.now().isoformat(
    timespec="seconds"
)
adata_all.uns["source_files"] = [
    record["path"].name for record in input_records
]

build_summary = {
    "dataset_id": "DS1_GSE103224",
    "geo_accession": "GSE103224",
    "n_samples": int(adata_all.obs["sample_id"].nunique()),
    "n_cells": int(adata_all.n_obs),
    "n_genes": int(adata_all.n_vars),
    "matrix_format": type(adata_all.X).__name__,
    "matrix_dtype": str(adata_all.X.dtype),
    "total_counts": int(adata_all.X.sum()),
    "maximum_count": int(adata_all.X.max()),
    "output_h5ad": str(OUTPUT_H5AD),
}

with open(
    AUDIT_DIR / "GSE103224_raw_build_summary.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(build_summary, handle, indent=2)

print(adata_all)
print(adata_all.obs["sample_id"].value_counts())
print(build_summary)

In [ ]:
temporary_output = OUTPUT_H5AD.with_suffix(".tmp.h5ad")
if temporary_output.exists():
    temporary_output.unlink()

adata_all.write_h5ad(temporary_output, compression="gzip")
temporary_output.replace(OUTPUT_H5AD)

with sc.read_h5ad(OUTPUT_H5AD, backed="r") as check:
    if check.shape != (EXPECTED_TOTAL_CELLS, EXPECTED_N_GENES):
        raise AssertionError(
            f"Saved-file validation failed: observed {check.shape}"
        )

print("Saved and validated:", OUTPUT_H5AD)